In [1]:
import pandas as pd
import numpy as np
from cdc_ml.config import POLLS_PROCESSED,CUSTOMER_CLASS_PROCESSED
import xgboost as xgb
from cdc_ml.modeling.train import temporal_split,train
from cdc_ml.features.build_features import assign_class_type
from sklearn.metrics import average_precision_score
import tqdm
from loguru import logger

2026-05-16 15:45:02.396 | INFO     | cdc_ml.config:<module>:12 - PROJ_ROOT path is: C:\Users\zhiju\Desktop\cdc_ml


In [2]:
df = pd.read_parquet(POLLS_PROCESSED)
df_class = pd.read_parquet(CUSTOMER_CLASS_PROCESSED)

In [3]:
df= assign_class_type(df,df_class)

In [4]:
df.sort_values(by='cycle_start').tail()

,id,username,cycle_start,cycle_end,polling_at,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle,class_type,is_one_team
31951,31951,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-22 08:00:00+08:00,0,4,21,1,0,4,22,2,8,32.0,0,1
31950,31950,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-22 07:00:00+08:00,0,4,21,1,0,4,22,2,7,31.0,0,1
31949,31949,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-22 06:00:00+08:00,0,4,21,1,0,4,22,2,6,30.0,0,1
31977,31977,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-23 10:00:00+08:00,0,4,21,1,0,4,23,3,10,58.0,0,1
32034,32034,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-25 19:00:00+08:00,0,4,21,1,0,4,25,5,19,115.0,0,1


In [5]:
df_train , df_test = temporal_split(df)

In [6]:
df_train

,id,username,cycle_start,cycle_end,polling_at,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle,class_type,is_one_team
0,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 09:15:00+08:00,0,8,19,1,9,8,19,1,9,0.0,0,0
1,1,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 10:15:00+08:00,0,8,19,1,9,8,19,1,10,1.0,0,0
2,2,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 11:15:00+08:00,0,8,19,1,9,8,19,1,11,2.0,0,0
3,3,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 12:15:00+08:00,0,8,19,1,9,8,19,1,12,3.0,0,0
4,4,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 13:15:00+08:00,0,8,19,1,9,8,19,1,13,4.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32487,32487,addity,2025-10-16 00:00:00+08:00,2025-10-17 20:00:00+08:00,2025-10-17 16:00:00+08:00,0,10,16,3,0,10,17,4,16,40.0,1,1
32488,32488,addity,2025-10-16 00:00:00+08:00,2025-10-17 20:00:00+08:00,2025-10-17 17:00:00+08:00,0,10,16,3,0,10,17,4,17,41.0,1,1
32489,32489,addity,2025-10-16 00:00:00+08:00,2025-10-17 20:00:00+08:00,2025-10-17 18:00:00+08:00,1,10,16,3,0,10,17,4,18,42.0,1,1
32490,32490,addity,2025-10-16 00:00:00+08:00,2025-10-17 20:00:00+08:00,2025-10-17 19:00:00+08:00,0,10,16,3,0,10,17,4,19,43.0,1,1


In [7]:
df_train["has_booking"].mean()

np.float64(0.011837066264361146)

In [8]:
df_test["has_booking"].mean()

np.float64(0.013552175877126938)

In [9]:
seed = np.random.randint(0,100)
oof_xgb,oof_marg,oof_const=train(df_train,["cycle_start_month","cycle_start_day","cycle_start_dow","cycle_start_hour","polling_month","polling_day","hours_into_cycle","class_type","is_one_team"],seed)

      polling_dow  polling_hour
9052            3            22
fold 0: train n=  4308 (2025-08-12 → 2025-10-07)  val n=  4308 (2025-10-07 → 2025-11-02)  
  train_pos=0.009  val_pos=0.013  
  marg_brier=0.0133  marg_pr=0.0202  
  xgb_brier_val=0.0130  xgb_pr_val=0.0277  
  xgb_brier_tr=0.0088  xgb_pr_tr=0.0437  

fold 1: train n=  8616 (2025-08-12 → 2025-11-02)  val n=  4308 (2025-11-02 → 2025-11-18)  
  train_pos=0.011  val_pos=0.008  
  marg_brier=0.0081  marg_pr=0.0115  
  xgb_brier_val=0.0079  xgb_pr_val=0.0142  
  xgb_brier_tr=0.0109  xgb_pr_tr=0.0353  

fold 2: train n= 12924 (2025-08-12 → 2025-11-18)  val n=  4308 (2025-11-18 → 2025-12-08)  
  train_pos=0.010  val_pos=0.014  
  marg_brier=0.0141  marg_pr=0.0306  
  xgb_brier_val=0.0140  xgb_pr_val=0.0383  
  xgb_brier_tr=0.0099  xgb_pr_tr=0.0285  

fold 3: train n= 17232 (2025-08-12 → 2025-12-08)  val n=  4308 (2025-12-08 → 2026-01-08)  
  train_pos=0.011  val_pos=0.016  
  marg_brier=0.0154  marg_pr=0.0347  
  xgb_brier_val=0.0

In [10]:
# from pathlib import Path
# from sklearn.model_selection import permutation_test_score
# from cdc_ml.features.build_features import drop_meta_high_card_cols
# from cdc_ml.modeling.train import time_ordered_kfold
# from cdc_ml.config import STAGE_1_PROCESSED

# def perm(
#     data_path: Path = STAGE_1_PROCESSED,
#     n_permutations: int = 1000,
#     seed: int = 42,
# ):
#     df = pd.read_parquet(data_path)
#     df_train, _ = temporal_split(df)

#     y = df_train["has_booking"].to_numpy()
#     X = drop_meta_high_card_cols(df_train).drop(columns=["has_booking"]+["cycle_start_month","cycle_start_day","cycle_start_dow","cycle_start_hour","polling_month","polling_day","hours_into_cycle","class_type","is_one_team"])

#     splits = list(time_ordered_kfold(df_train, time_col="cycle_start"))

#     clf = xgb.XGBClassifier(
#         n_estimators=500,
#         objective="binary:logistic",
#         eval_metric="aucpr",
#         learning_rate=0.03,
#         max_depth=1,
#         min_child_weight=10,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_lambda=1.0,
#         tree_method="hist",
#         device="cuda",
#         random_state=seed,
#         n_jobs=1,                  # let permutation_test_score parallelize
#     )

#     score, perm_scores, pvalue = permutation_test_score(
#         clf, X, y,
#         scoring="average_precision",
#         cv=splits,                 # pre-computed list of (train, val) index pairs
#         n_permutations=n_permutations,
#         n_jobs=-1,
#         random_state=seed,
#         verbose=1,
#     )

#     print(
#         f"\ntrue PR-AUC = {score:.4f}\n"
#         f"null        = {perm_scores.mean():.4f} ± {perm_scores.std():.4f}\n"
#         f"p-value     = {pvalue:.4f}  (n_perm={n_permutations})"
#     )

# perm(STAGE_1_PROCESSED)